# 06 — Facebook Attention, Policy Frames, Topic Structure, and Engagement


    **Research objectives.** Describe monthly attention in the consistent 2024–2026 `datacenter` corpus, identify prevalent policy frames and topics during April–August 2026, and estimate which terms are associated with above-owner-median engagement.

    **Inputs.** Prepared longitudinal and April–August 2026 `datacenter` files from notebook 05.

    **Methods.** Aggregate monthly unique-text counts and a three-month moving average; apply six non-mutually-exclusive regular-expression frames to the 2026 subset; compare count-vector LDA solutions with four through eight topics; and estimate a TF-IDF unigram/bigram logistic regression using normalized-text-deduplicated 2026 posts. The engagement outcome equals one when a post exceeds its owner's median among owners with at least five posts; ties remain in the reference category. Both a stratified post holdout and an owner-group holdout are reported.

    **Outputs.** Longitudinal and frame figures, topic tables and representative posts, engagement metrics, and coefficient figure in `../output/`.

In [1]:
from __future__ import annotations

import json

import math

import re

from pathlib import Path

import matplotlib

import matplotlib.dates as mdates

import matplotlib.pyplot as plt

import numpy as np

import pandas as pd

from scipy.stats import spearmanr

from sklearn.compose import ColumnTransformer

from sklearn.decomposition import LatentDirichletAllocation

from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

from sklearn.model_selection import (
    GroupShuffleSplit,
    StratifiedKFold,
    cross_val_predict,
    train_test_split,
)

from sklearn.pipeline import Pipeline

from sklearn.preprocessing import OneHotEncoder, StandardScaler

matplotlib.use("Agg")

SEED = 149

COLORS = {
    "restrictive": "#D55E00",
    "neutral": "#7A7A7A",
    "supportive": "#009E73",
    "blue": "#0072B2",
    "orange": "#E69F00",
    "purple": "#CC79A7",
    "sky": "#56B4E9",
}

FRAME_PATTERNS = {
    "Energy and utility costs": [
        r"\belectric(?:ity|al)\b", r"\bpower\b", r"\bgrid\b", r"\butilit(?:y|ies)\b",
        r"\bratepayers?\b", r"\brates?\b", r"\bmegawatts?\b", r"\btransmission\b",
        r"\binterconnection\b", r"\benergy\b",
    ],
    "Water and environmental effects": [
        r"\bwater\b", r"\benvironment(?:al)?\b", r"\bemissions?\b", r"\bcarbon\b",
        r"\bpollution\b", r"\bclimate\b", r"\bsustainab(?:le|ility)\b", r"\baquifer\b",
    ],
    "Economic development and jobs": [
        r"\bjobs?\b", r"\bemployment\b", r"\beconomic development\b", r"\binvest(?:ment|s|ed)\b",
        r"\bbusiness(?:es)?\b", r"\bconstruction\b", r"\bgrowth\b", r"\brevenue\b",
    ],
    "Taxes, incentives, and subsidies": [
        r"\btax(?:es|ation)?\b", r"\btax (?:credit|break|exemption)s?\b", r"\bincentives?\b",
        r"\bsubsid(?:y|ies)\b", r"\babatement\b", r"\bpublic funds?\b",
    ],
    "Regulation and community control": [
        r"\bregulat(?:ion|e|ed|ory)\b", r"\bzoning\b", r"\bpermits?\b", r"\bordina(?:nce|nces)\b",
        r"\bmoratorium\b", r"\bbans?\b", r"\bcommunity\b", r"\blocal control\b",
        r"\bpublic hearings?\b", r"\bdisclos(?:ure|e)\b", r"\btransparency\b",
    ],
    "AI growth and technological competition": [
        r"\bartificial intelligence\b", r"\bAI\b", r"\bcloud\b", r"\bcompute\b",
        r"\bdigital infrastructure\b", r"\btechnology\b", r"\binnovation\b", r"\bhyperscale\b",
    ],
}

def save_table(frame: pd.DataFrame, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False)
    print(f"Saved {len(frame):,} rows -> {path}")

def save_figure(fig: plt.Figure, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"Saved figure -> {path}")

def slug(value: str) -> str:
    return re.sub(r"[^a-z0-9]+", "_", value.lower()).strip("_")

def classification_metrics(y_true, y_pred, y_prob, model_name: str) -> dict:
    return {
        "model": model_name,
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, y_prob) if len(np.unique(y_true)) == 2 else np.nan,
        "n": len(y_true), "positive_n": int(np.sum(y_true)),
    }

def _read_processed(path: Path) -> pd.DataFrame:
    frame = pd.read_csv(path, low_memory=False)
    if "creation_time" in frame:
        frame["creation_time"] = pd.to_datetime(frame["creation_time"], errors="coerce", utc=True)
    bool_columns = [c for c in frame if c == "policy_related" or c.startswith("frame_")]
    for column in bool_columns:
        if frame[column].dtype != bool:
            frame[column] = frame[column].astype(str).str.lower().eq("true")
    return frame

def _owner_language_model(frame: pd.DataFrame, group_holdout: bool = False):
    counts = frame["post_owner.id"].value_counts()
    eligible = frame.loc[frame["post_owner.id"].isin(counts[counts.ge(5)].index)].copy()
    owner_median = eligible.groupby("post_owner.id")["total_engagement"].transform("median")
    eligible["owner_median_engagement"] = owner_median
    eligible["above_owner_median"] = eligible.total_engagement.gt(owner_median).astype(int)
    X = eligible.normalized_text.fillna("")
    y = eligible.above_owner_median
    groups = eligible["post_owner.id"]
    vectorizer = TfidfVectorizer(
        stop_words="english", ngram_range=(1, 2), min_df=5, max_df=.95,
        max_features=5000, sublinear_tf=True,
    )
    minority_share = min(y.mean(), 1-y.mean())
    class_weight = "balanced" if minority_share < .40 else None
    model = LogisticRegression(max_iter=2000, class_weight=class_weight, random_state=SEED)
    if group_holdout:
        splitter = GroupShuffleSplit(n_splits=1, test_size=.25, random_state=SEED)
        train_idx, test_idx = next(splitter.split(X, y, groups))
    else:
        train_idx, test_idx = train_test_split(
            np.arange(len(eligible)), test_size=.25, random_state=SEED, stratify=y
        )
    X_train = vectorizer.fit_transform(X.iloc[train_idx])
    X_test = vectorizer.transform(X.iloc[test_idx])
    model.fit(X_train, y.iloc[train_idx])
    pred = model.predict(X_test)
    prob = model.predict_proba(X_test)[:, 1]
    majority = int(y.iloc[train_idx].mean() >= .5)
    base_pred = np.repeat(majority, len(test_idx))
    base_prob = np.repeat(y.iloc[train_idx].mean(), len(test_idx))
    suffix = "owner-group holdout" if group_holdout else "stratified post holdout"
    metrics = pd.DataFrame([
        classification_metrics(y.iloc[test_idx], base_pred, base_prob, f"Majority baseline ({suffix})"),
        classification_metrics(y.iloc[test_idx], pred, prob, f"TF-IDF logistic regression ({suffix})"),
    ])
    coefficients = pd.DataFrame({
        "term": vectorizer.get_feature_names_out(),
        "coefficient": model.coef_[0],
    }).sort_values("coefficient")
    diagnostics = {
        "eligible_posts": len(eligible), "eligible_owners": eligible["post_owner.id"].nunique(),
        "positive_posts": int(y.sum()), "negative_or_tied_posts": int((1-y).sum()),
        "ties_at_owner_median": int(eligible.total_engagement.eq(owner_median).sum()),
        "train_posts": len(train_idx), "test_posts": len(test_idx),
    }
    return metrics, coefficients, diagnostics

def run_facebook_discourse(root: Path) -> dict:
    longitudinal = _read_processed(root / "data/processed/facebook_datacenter_longitudinal_2024_2026.csv.gz")
    discourse_2026 = _read_processed(root / "data/processed/facebook_datacenter_discourse_2026.csv.gz")
    discourse_policy = _read_processed(root / "data/processed/facebook_datacenter_discourse_2026_policy.csv.gz")
    discourse_text = _read_processed(root / "data/processed/facebook_datacenter_discourse_2026_text_deduplicated.csv.gz")
    discourse_policy_text = _read_processed(root / "data/processed/facebook_datacenter_discourse_2026_policy_text_deduplicated.csv.gz")
    print(f"Longitudinal datacenter corpus: {len(longitudinal):,} rows")
    print(f"Apr-Aug 2026 datacenter discourse: {len(discourse_2026):,} rows; policy subset: {len(discourse_policy):,}")

    longitudinal["month"] = longitudinal.creation_time.dt.tz_localize(None).dt.to_period("M").astype(str)
    monthly = longitudinal.groupby("month").agg(
        relevant_post_count=("id", "nunique"), policy_related_post_count=("policy_related", "sum"),
        unique_owners=("post_owner.id", "nunique"), unique_normalized_texts=("normalized_text", "nunique"),
        median_engagement=("total_engagement", "median"), total_engagement=("total_engagement", "sum"),
        duplicate_rate=("duplicate_count", lambda x: np.mean(pd.to_numeric(x, errors="coerce").fillna(1).gt(1))),
    ).reset_index()
    full_months = pd.period_range("2024-01", "2026-08", freq="M").astype(str)
    calendar = pd.DataFrame({"month": full_months})
    print(
        f"Before calendar merge: calendar={len(calendar):,} rows; "
        f"observed monthly summary={len(monthly):,} rows"
    )
    monthly = calendar.merge(
        monthly, on="month", how="left", validate="one_to_one",
        indicator="_calendar_merge",
    )
    matched_months = int(monthly["_calendar_merge"].eq("both").sum())
    print(
        f"After calendar merge: {len(monthly):,} rows; "
        f"matched months={matched_months:,}; explicit zero months={len(monthly)-matched_months:,}"
    )
    monthly = monthly.drop(columns="_calendar_merge")
    count_cols = ["relevant_post_count","policy_related_post_count","unique_owners","unique_normalized_texts","total_engagement"]
    monthly[count_cols] = monthly[count_cols].fillna(0)
    monthly["unique_texts_3_month_mean"] = monthly.unique_normalized_texts.rolling(3, min_periods=1).mean()
    monthly["policy_share"] = monthly.policy_related_post_count.div(monthly.relevant_post_count.replace(0,np.nan))
    save_table(monthly, root / "output/tables/facebook_longitudinal_monthly.csv")
    dates = pd.to_datetime(monthly.month)
    fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True, gridspec_kw={"height_ratios":[2,1]})
    axes[0].bar(dates, monthly.unique_normalized_texts, width=22, color=COLORS["sky"], alpha=.42, label="Unique normalized texts")
    axes[0].plot(dates, monthly.unique_texts_3_month_mean, color=COLORS["blue"], lw=2.5, label="3-month moving average")
    axes[0].plot(dates, monthly.unique_owners, color=COLORS["purple"], lw=1.5, label="Unique owners")
    axes[0].set_ylabel("Monthly unique count")
    axes[0].set_title(f"Monthly Facebook Attention in Posts Returned by the 'datacenter' Query (n={len(longitudinal):,})\nUnique normalized texts reduce the influence of repeated and syndicated content")
    axes[0].grid(axis="y", alpha=.2)
    axes[0].legend(ncol=3, frameon=False)
    axes[1].bar(dates, monthly.policy_related_post_count, width=22, color=COLORS["orange"], alpha=.65, label="Policy-related posts")
    share_axis = axes[1].twinx()
    share_axis.plot(dates, 100*monthly.policy_share, color=COLORS["restrictive"], marker="o", ms=3, label="Policy share")
    axes[1].set_ylabel("Policy-post count")
    share_axis.set_ylabel("Policy share of relevant posts (%)")
    axes[1].grid(axis="y", alpha=.2)
    lines, labels = axes[1].get_legend_handles_labels(); lines2, labels2 = share_axis.get_legend_handles_labels()
    axes[1].legend(lines+lines2, labels+labels2, ncol=2, frameon=False, loc="upper left")
    for ax in axes:
        ax.axvspan(pd.Timestamp("2026-08-01"), pd.Timestamp("2026-09-01"), color="0.75", alpha=.25)
    axes[1].annotate("Partial month", xy=(pd.Timestamp("2026-08-09"), monthly.policy_related_post_count.iloc[-1]), xytext=(-65,22), textcoords="offset points", arrowprops={"arrowstyle":"->","color":"0.35"}, fontsize=9)
    axes[1].set_xlabel("Month")
    axes[1].xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    axes[1].xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    fig.tight_layout()
    save_figure(fig, root / "output/figures/facebook_longitudinal_attention.png")

    coverage_audit = monthly[["month","relevant_post_count","policy_related_post_count","unique_owners"]].copy()
    coverage_audit["month_over_month_percent_change"] = coverage_audit.relevant_post_count.replace(0,np.nan).pct_change(fill_method=None)*100
    coverage_audit["abrupt_absolute_change_over_100_percent"] = coverage_audit.month_over_month_percent_change.abs().gt(100)
    repeated_counts = coverage_audit.relevant_post_count.value_counts()
    coverage_audit["possible_repeated_exact_count_cap"] = coverage_audit.relevant_post_count.map(repeated_counts).ge(3)
    coverage_audit["partial_month"] = coverage_audit.month.eq("2026-08")
    save_table(coverage_audit, root / "output/tables/facebook_longitudinal_discontinuity_flags.csv")

    frame_rows = []
    flag_columns = [f"frame_{slug(name)}" for name in FRAME_PATTERNS]
    for name, column in zip(FRAME_PATTERNS, flag_columns):
        subset = discourse_policy.loc[discourse_policy[column]]
        frame_rows.append({
            "frame": name, "post_count": len(subset), "unique_owners": subset["post_owner.id"].nunique(),
            "percent_of_policy_posts": 100*len(subset)/len(discourse_policy) if len(discourse_policy) else np.nan,
            "median_engagement": subset.total_engagement.median(), "mean_log_engagement": subset.log_engagement.mean(),
            "p75_engagement": subset.total_engagement.quantile(.75), "percent_zero_engagement": 100*subset.total_engagement.eq(0).mean(),
            "duplicate_rate": subset.duplicate_count.gt(1).mean(),
        })
    frame_summary = pd.DataFrame(frame_rows).sort_values("post_count", ascending=False)
    save_table(frame_summary, root / "output/tables/facebook_2026_frame_summary.csv")
    overlap = discourse_policy[flag_columns].astype(int).T.dot(discourse_policy[flag_columns].astype(int))
    overlap.index = list(FRAME_PATTERNS)
    overlap.columns = list(FRAME_PATTERNS)
    save_table(overlap.reset_index(names="frame"), root / "output/tables/facebook_2026_frame_overlap.csv")
    fig, axes = plt.subplots(1, 2, figsize=(14, 6))
    ordered = frame_summary.sort_values("post_count")
    axes[0].barh(ordered.frame, ordered.percent_of_policy_posts, color=COLORS["blue"])
    axes[0].set_xlabel("Percent of policy posts (frames may overlap)")
    axes[0].set_title("Policy-Frame Prevalence")
    axes[1].barh(ordered.frame, ordered.median_engagement, color=COLORS["orange"])
    axes[1].set_xlabel("Median reactions + comments + shares")
    axes[1].set_title("Median Post Engagement")
    for ax in axes:
        ax.grid(axis="x", alpha=.2)
    fig.suptitle(f"Policy Frames in Facebook Posts Returned by the 'datacenter' Query, April 1–August 17, 2026\nPolicy-related posts: n={len(discourse_policy):,}; frame categories are non-mutually exclusive")
    for i, row in enumerate(ordered.itertuples()):
        axes[0].text(row.percent_of_policy_posts+.3, i, f"n={row.post_count:,}", va="center", fontsize=8)
    fig.tight_layout()
    save_figure(fig, root / "output/figures/facebook_2026_frame_engagement.png")

    texts = discourse_policy_text.normalized_text.fillna("")
    min_df = max(5, int(len(texts)*.001))
    vectorizer = CountVectorizer(stop_words="english", ngram_range=(1,2), min_df=min_df, max_df=.85, max_features=5000)
    counts = vectorizer.fit_transform(texts)
    topic_diagnostics = []
    topic_models = {}
    for k in range(4, 9):
        lda = LatentDirichletAllocation(n_components=k, random_state=SEED, learning_method="batch", max_iter=12)
        lda.fit(counts)
        topic_models[k] = lda
        topic_diagnostics.append({"n_topics": k, "perplexity": lda.perplexity(counts), "log_likelihood": lda.score(counts)})
    topic_diag = pd.DataFrame(topic_diagnostics)
    save_table(topic_diag, root / "output/tables/facebook_topic_model_diagnostics.csv")
    chosen_k = 6
    lda = topic_models[chosen_k]
    distributions = lda.transform(counts)
    primary = distributions.argmax(axis=1)
    terms = vectorizer.get_feature_names_out()
    interpreted_labels = {
        1: "AI Infrastructure and Technical Commentary",
        2: "Claremore Government Transparency, Litigation, and Recall Mobilization",
        3: "Local Development, Public Finance, and Mixed Commentary",
        4: "Water, Environmental, and Community Impacts",
        5: "State and Local Restrictions, Resource Requirements, and Texas Politics",
        6: "Federal Policy, Business Reporting, and Peripheral AI Commentary",
    }
    topic_rows, representative_rows = [], []
    for topic in range(chosen_k):
        top_terms = terms[np.argsort(lda.components_[topic])[-12:][::-1]].tolist()
        member_idx = np.where(primary == topic)[0]
        top_idx = np.argsort(distributions[:, topic])[-5:][::-1]
        topic_rows.append({
            "topic": topic+1, "researcher_interpreted_label": interpreted_labels[topic+1],
            "top_terms": " | ".join(top_terms), "post_count_primary_topic": len(member_idx),
            "median_engagement": discourse_policy_text.iloc[member_idx].total_engagement.median() if len(member_idx) else np.nan,
        })
        for rank, idx in enumerate(top_idx, 1):
            representative_rows.append({
                "topic": topic+1, "rank": rank, "topic_probability": distributions[idx, topic],
                "post_id": discourse_policy_text.iloc[idx].id, "text": str(discourse_policy_text.iloc[idx].text)[:1000],
                "total_engagement": discourse_policy_text.iloc[idx].total_engagement,
            })
    topic_summary = pd.DataFrame(topic_rows)
    save_table(topic_summary, root / "output/tables/facebook_topic_summary.csv")
    save_table(pd.DataFrame(representative_rows), root / "output/tables/facebook_topic_representative_posts.csv")
    topic_week = discourse_policy_text[["creation_time"]].copy()
    topic_week["week"] = topic_week.creation_time.dt.tz_localize(None).dt.to_period("W").astype(str)
    topic_week["topic"] = primary + 1
    weekly = topic_week.groupby(["week","topic"]).size().rename("post_count").reset_index()
    save_table(weekly, root / "output/tables/facebook_topic_weekly_prevalence.csv")

    metrics, coefficients, diagnostics = _owner_language_model(discourse_text, group_holdout=False)
    group_metrics, _, group_diagnostics = _owner_language_model(discourse_text, group_holdout=True)
    all_metrics = pd.concat([metrics, group_metrics], ignore_index=True)
    save_table(all_metrics, root / "output/tables/engagement_model_metrics.csv")
    strongest = pd.concat([coefficients.head(20), coefficients.tail(20)]).drop_duplicates().sort_values("coefficient")
    save_table(strongest, root / "output/tables/engagement_language_coefficients.csv")
    fig, ax = plt.subplots(figsize=(10, 9))
    plot_coef = pd.concat([coefficients.head(15), coefficients.tail(15)]).sort_values("coefficient")
    ax.barh(plot_coef.term, plot_coef.coefficient, color=np.where(plot_coef.coefficient.ge(0), COLORS["supportive"], COLORS["restrictive"]))
    ax.axvline(0, color="black", lw=1)
    ax.set_xlabel("Logistic-regression coefficient")
    ax.set_title("Terms Associated with Above-Owner-Median Engagement\nFacebook posts returned by the 'datacenter' query, April 1–August 17, 2026; coefficients are noncausal associations")
    ax.grid(axis="x", alpha=.2)
    fig.tight_layout()
    save_figure(fig, root / "output/figures/engagement_language_coefficients.png")
    diagnostics.update({f"group_{k}": v for k,v in group_diagnostics.items()})
    save_table(pd.DataFrame([diagnostics]), root / "output/tables/engagement_model_diagnostics.csv")
    return {
        "longitudinal_rows": len(longitudinal), "discourse_2026_policy_rows": len(discourse_policy),
        "chosen_topics": chosen_k, "engagement_model": all_metrics.to_dict("records"),
        "engagement_diagnostics": diagnostics,
    }

ROOT = Path("..")
print("Random seed:", SEED)

Random seed: 149


In [2]:
results = run_facebook_discourse(ROOT)
results

Longitudinal datacenter corpus: 24,915 rows
Apr-Aug 2026 datacenter discourse: 10,597 rows; policy subset: 1,995
Before calendar merge: calendar=32 rows; observed monthly summary=32 rows
After calendar merge: 32 rows; matched months=32; explicit zero months=0
Saved 32 rows -> ../output/tables/facebook_longitudinal_monthly.csv


Saved figure -> ../output/figures/facebook_longitudinal_attention.png
Saved 32 rows -> ../output/tables/facebook_longitudinal_discontinuity_flags.csv
Saved 6 rows -> ../output/tables/facebook_2026_frame_summary.csv
Saved 6 rows -> ../output/tables/facebook_2026_frame_overlap.csv


Saved figure -> ../output/figures/facebook_2026_frame_engagement.png


Saved 5 rows -> ../output/tables/facebook_topic_model_diagnostics.csv
Saved 6 rows -> ../output/tables/facebook_topic_summary.csv
Saved 30 rows -> ../output/tables/facebook_topic_representative_posts.csv
Saved 120 rows -> ../output/tables/facebook_topic_weekly_prevalence.csv


Saved 4 rows -> ../output/tables/engagement_model_metrics.csv
Saved 40 rows -> ../output/tables/engagement_language_coefficients.csv


Saved figure -> ../output/figures/engagement_language_coefficients.png
Saved 1 rows -> ../output/tables/engagement_model_diagnostics.csv


{'longitudinal_rows': 24915,
 'discourse_2026_policy_rows': 1995,
 'chosen_topics': 6,
 'engagement_model': [{'model': 'Majority baseline (stratified post holdout)',
   'accuracy': 0.5692963752665245,
   'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0,
   'roc_auc': 0.5,
   'n': 938,
   'positive_n': 404},
  {'model': 'TF-IDF logistic regression (stratified post holdout)',
   'accuracy': 0.5767590618336887,
   'precision': 0.5169082125603864,
   'recall': 0.26485148514851486,
   'f1': 0.3502454991816694,
   'roc_auc': 0.5586364816256907,
   'n': 938,
   'positive_n': 404},
  {'model': 'Majority baseline (owner-group holdout)',
   'accuracy': 0.575,
   'precision': 0.0,
   'recall': 0.0,
   'f1': 0.0,
   'roc_auc': 0.5,
   'n': 840,
   'positive_n': 357},
  {'model': 'TF-IDF logistic regression (owner-group holdout)',
   'accuracy': 0.575,
   'precision': 0.5,
   'recall': 0.22969187675070027,
   'f1': 0.31477927063339733,
   'roc_auc': 0.5477437351752295,
   'n': 840,
   'positive_n':

## Interpretation, inferential scope, and limitations

Counts refer only to the `datacenter` query and cannot be called total Facebook discussion. The corpus misses spaced-only usage. Frame matches overlap and require human validation. LDA labels are researcher interpretations of terms and representative posts. Engagement is not public opinion, exposure, persuasion, or support; follower counts are unavailable, pages are not a verified politician sample, and coefficient language is associational.